# Pothole & Road Damage Detection: Colab (T4 GPU)

Runs the pretrained EfficientNetV2-S classifier (crack / normal / pothole) on your road videos.
**Before you start:** `Runtime > Change runtime type > T4 GPU`, then run the cells top to bottom.

Repo: https://github.com/mourishantony/pothole-cnn (private: you need a read-only token, see COLAB_SETUP.md)

## 1. Check the GPU

In [ ]:
import tensorflow as tf
!nvidia-smi -L
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", gpus)
assert gpus, "No GPU found: Runtime > Change runtime type > T4 GPU, then reconnect and re-run."

## 2. Get the code and pretrained weights

In [ ]:
import os, getpass
REPO = "mourishantony/pothole-cnn"
if not os.path.exists("pothole-cnn"):
    # The repo is private: paste a read-only GitHub token (see COLAB_SETUP.md). Leave empty if the repo is public.
    token = getpass.getpass("GitHub token (empty if public): ").strip()
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    !git clone -q {url}
    !git -C pothole-cnn remote set-url origin https://github.com/{REPO}.git   # do not keep the token on disk
    del token, url
%cd pothole-cnn
size_mb = os.path.getsize("models/final.keras") / 1e6
print(f"models/final.keras = {size_mb:.0f} MB")
assert size_mb > 50, "Weights file looks wrong (expected ~83 MB)"


## 3. Install the few extra packages
Colab already has TensorFlow and OpenCV. We only add Flask and ffmpeg (for browser-playable output video).

In [ ]:
!pip install -q flask imageio-ffmpeg

## 4. Load the model
If this fails with a `could not be deserialized` / `input_axes` error, run `!pip install -U keras`, then
`Runtime > Restart session` and re-run from step 1.

In [ ]:
import sys
sys.path.insert(0, ".")
import app            # loads models/final.keras and defines process()
print("Model loaded. Classes:", app.CLASSES)

## 5. Upload a road video

In [ ]:
from google.colab import files
uploaded = files.upload()          # pick an .mp4 / .avi / .mov / .mkv / .webm
video_path = next(iter(uploaded))
print("Uploaded:", video_path)

## 6. Analyse it
`every_sec` = how often a frame is checked (0.5 = every half second). Raise it (1.0 or 2.0) for long videos to go faster,
at the risk of missing a short pothole.

In [ ]:
import pathlib, time
from IPython.display import Video, Image, display

out_dir = pathlib.Path("results"); out_dir.mkdir(exist_ok=True)
t0 = time.time()
r = app.process(pathlib.Path(video_path), out_dir, every_sec=0.5)
print(f"Done in {time.time()-t0:.0f}s ->", r)
print()
print("==> POTHOLE DETECTED" if r["pothole_found"] else "==> No pothole detected")
display(Video(str(out_dir / "annotated.mp4"), embed=True, width=720))
if r["has_best"]:
    print("Strongest pothole frame:")
    display(Image(str(out_dir / "best.jpg")))

## 7. (Optional) Download the annotated video

In [ ]:
files.download("results/annotated.mp4")

## 8. (Optional) Launch the website inside Colab
Starts the Flask site and shows it in an iframe below, so you can upload videos through the web page.

In [ ]:
import threading
from google.colab import output

threading.Thread(
    target=lambda: app.app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False, threaded=False),
    daemon=True,
).start()
output.serve_kernel_port_as_iframe(5000, height=800)

---
## Optional: retrain from scratch on the T4
Skip this unless you want to retrain. It needs a **Kaggle API token** (kaggle.com > Settings > API > Create New Token) and
downloads about 11 GB. Colab sessions can disconnect, so save checkpoints to Google Drive (step A).
Do **not** share the token file or commit it anywhere.

### A. Mount Google Drive (to keep checkpoints)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/pothole_models

### B. Add your Kaggle token and install training packages

In [ ]:
import os, shutil
from google.colab import files
print("Select your kaggle.json")
up = files.upload()
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
shutil.move(next(iter(up)), os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
!pip install -q kaggle scikit-learn matplotlib seaborn pillow

### C. Download and prepare the data (about 11 GB, takes a while)

In [ ]:
!python download_data.py lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes
!python prepare_data.py

In [ ]:
# RDD2022: download as a zip (do not unzip), then build the combined crop dataset
!mkdir -p data/raw/rdd2022
!kaggle datasets download -d aliabdelmenam/rdd-2022 -p data/raw/rdd2022
!python prepare_rdd.py

### D. Train
Stage 1 trains the new head and fine-tunes the top layers; stage 2 resumes and fine-tunes the whole network on data
with the original-domain crops oversampled 3x. Each stage can take over an hour on a T4.

In [ ]:
!DATA_DIR=processed_v2 BACKBONE=efficientnetv2s TAG=effv2s_v2 BATCH=32 HEAD_EP=6 FT_EP=20 FT_LAYERS=120 python train.py
!cp models/best_effv2s_v2.keras /content/drive/MyDrive/pothole_models/

In [ ]:
# build processed_v3 = processed_v2 + 2 extra copies of the original-domain training crops
import shutil, glob, os
shutil.copytree("data/processed_v2", "data/processed_v3", dirs_exist_ok=True)
for c in ("crack", "normal", "pothole"):
    for f in glob.glob(f"data/processed_v2/train/{c}/orig_*.jpg"):
        b = os.path.basename(f)
        for k in (1, 2):
            shutil.copy2(f, f"data/processed_v3/train/{c}/dup{k}_{b}")

In [ ]:
!DATA_DIR=processed_v3 BACKBONE=efficientnetv2s TAG=effv2s_v3 RESUME=best_effv2s_v2.keras BATCH=32 FT_EP=15 FT_LAYERS=600 FT_LR=3e-5 python train.py
!cp models/best_effv2s_v3.keras /content/drive/MyDrive/pothole_models/
!cat outputs/report_effv2s_v3_test_orig.txt outputs/report_effv2s_v3_test_rdd.txt

In [ ]:
# optional: strip the optimizer state (210 MB -> ~83 MB) and use it as the model for step 4
import tensorflow as tf
m = tf.keras.models.load_model("models/best_effv2s_v3.keras", compile=False)
m.save("models/final.keras", include_optimizer=False)